# Prep

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import sys
import os

In [ ]:
plot_run = True

In [ ]:
custom_palette = sns.color_palette()[:4] 
manhatti_palette = {}
for c in range(1, 23):
    manhatti_palette[c] = custom_palette[c % len(custom_palette)]

renamer = {
    'chrom': '#CHROM',
    'pos': 'POS',
    'P': 'P',
    'se_ad_risk_by_proxy': 'SE'
}

font_kwargs={
    'fontsize':18
}
sns.set_theme(font_scale=1.2, context='paper', style='white', palette='tab10')
# set default stuff
plink_keys=['#CHROM', "POS"]
meta_keys=['CHR', "BP"]
suggestive_t=1e-5
gw_t=1.494299248e-7


In [ ]:
def get_associaTR_results(file_path, cut=True, cut_threshold=10, p_name='p_ad_risk_by_proxy', rename_chr=True):
    df = pd.read_csv(file_path, sep="\t").rename(columns={
        p_name:'P'
    })
    initial = df.shape[0]
    df = df.dropna(axis=0)
    df = df[df['locus_filtered'] == 'False']
    after = df.shape[0]
    print(f'drop {initial-after} rows because of NA values')
    df = df.astype({'P':float})
    df.loc[df.P<sys.float_info.min, 'P']=sys.float_info.min
    min_ct = df[df.P == sys.float_info.min].shape[0]
    if min_ct > 0:
        print(f'set {min_ct} P values to maximum possible min of {sys.float_info.min}')
    df['-log10'] = -np.log10(df.P)

    if cut:
        df=df[df['-log10'] < cut_threshold]
    if rename_chr:
        df['chrom'] = df['chrom'].str.replace('chr', '').astype(int)
    return df.sort_values(by=['chrom', 'pos']).reset_index(drop=True)
    
def get_results(file_path, is_meta=False, cut=True, cut_threshold=10):
    df = pd.read_csv(file_path, sep="\t")
    initial = df.shape[0]
    df = df.dropna(axis=0)
    after = df.shape[0]
    print(f'drop {initial-after} rows because of NA values')
    df = df.astype({'P':float})
    df.loc[df.P<sys.float_info.min, 'P']=sys.float_info.min

    min_ct = df[df.P == sys.float_info.min].shape[0]
    if min_ct > 0:
        print(f'set {min_ct} P values to maximum possible min of {sys.float_info.min}')
    df['-log10'] = -np.log10(df.P)

    if not is_meta:
        df = df[df['TEST']=='ADD']

    if cut:
        df=df[df['-log10'] < cut_threshold]
    
    return df

In [ ]:
def manhatti(df, title, sort_keys=['#CHROM', "POS"], hue_key='#CHROM', style_key=None, y_steps=10, legend=None, 
             sort_key='-log10', sort_asc=True, plot_y_thresh=True, do_full=True, suggest_t=-np.log10(suggestive_t), p_sig_t=-np.log10(gw_t), palette=manhatti_palette):
    plot_df = df.copy(deep=True)
    plt.figure(figsize=(30,10))
    if do_full:
        chr_maxxs = plot_df.groupby('#CHROM')['POS'].max()
        chroms_sorted = sorted(chr_maxxs.keys())
        chr_starts = {chroms_sorted[0]: 0}
        for idx, c in enumerate(chroms_sorted[1:], start=1):
            prev_c = chroms_sorted[idx - 1]
            chr_starts[c] = chr_starts[prev_c] + chr_maxxs[prev_c] + 1
        plot_df['i'] = [p + chr_starts[c] for p,c in zip(plot_df['POS'], plot_df['#CHROM'])]
    else:
        plot_df = df.sort_values(['numeric_chr', sort_keys[1]])
        plot_df = plot_df.reset_index(drop=True); 
        plot_df['i'] = plot_df.index

    plot_df = plot_df.sort_values(by=sort_key, ascending=sort_asc)
    plot = sns.scatterplot(plot_df, x='i', y='-log10', hue=hue_key, legend=legend, linewidth=0.3, style=style_key, palette=palette)
    labels_df=plot_df.groupby(sort_keys[0])['i'].median()
    plot.set_xlabel('chr')
    plot.set_xticks(labels_df)
    plot.set_xticklabels(labels_df.index)
    max_x = plot_df.i.max()
    max_x = max_x + 0.01*max_x
    min_x = plot_df.i.min() - 0.01*max_x
    plt.xlim([min_x, max_x])

    max_y=int(max(plot_df['-log10'].max(), y_steps))+1
    step = int(max_y/y_steps)
    
    plot.set_yticks(range(1, max_y, step))
    if plot_y_thresh:
        plt.hlines(y=[p_sig_t, suggest_t], xmin=min_x, xmax=max_x, colors=['tab:red', 'k'], linestyles='dashed')

    plot.grid(axis='y')
    plot.set_title(title)
    plt.grid(axis='y')
    plt.title(title)
    plt.tight_layout()

    return plot.figure



def broken_manhatti(df, title, sort_keys=['#CHROM', "POS"], hue_key='#CHROM', style_key=None, legend=None, 
             sort_key='-log10', sort_asc=True, plot_y_thresh=True, do_full=True, suggest_t=5, p_sig_t=-np.log10(gw_t), plot_break=(30,100), kwargs={}):
    plot_df = df.copy(deep=True)
    f, (outlier_ax, plot_ax) = plt.subplots(2, 1, sharex=True, figsize=(30,10), height_ratios=[1,8])
    if do_full:
        chr_maxxs = plot_df.groupby('#CHROM')['POS'].max()
        chroms_sorted = sorted(chr_maxxs.keys())
        chr_starts = {chroms_sorted[0]: 0}
        for idx, c in enumerate(chroms_sorted[1:], start=1):
            prev_c = chroms_sorted[idx - 1]
            chr_starts[c] = chr_starts[prev_c] + chr_maxxs[prev_c] + 1
        plot_df['i'] = [p + chr_starts[c] for p,c in zip(plot_df['POS'], plot_df['#CHROM'])]
    else:
        plot_df = df.sort_values(sort_keys)
        plot_df = plot_df.reset_index(drop=True); 
        plot_df['i'] = plot_df.index

    plot_df = plot_df.sort_values(by=sort_key, ascending=sort_asc)
    plot = sns.scatterplot(plot_df, x='i', y='-log10', hue=hue_key, palette=manhatti_palette, legend=None, linewidth=0.3, style=style_key, ax=plot_ax, **kwargs)
    
    labels_df=plot_df.groupby(sort_keys[0])['i'].agg(['median', 'min', 'max'])
    # labels_df['mid'] = (labels_df['min'] + labels_df['max']) / 2 
    # labels_df['mid'] = labels_df.apply(lambda row: row['mid']/2 if row['mid'] == row['max'] else row['mid'], axis=1)
    
    labels_df['mid'] = (labels_df['max'].shift() + labels_df['max']) / 2
    labels_df.loc[labels_df['mid'].isna(), 'mid'] = labels_df['max'].iloc[0] / 2

    plot_ax.set_xticks(labels_df['max'])
    plot_ax.set_xticklabels([], minor=False)

    plot_ax.set_xticks(labels_df['mid'], minor=True)
    plot_ax.set_xticklabels(labels_df.index, minor=True)
    plot_ax.xaxis.grid(True)
    
    max_x = plot_df.i.max()
    max_x = max_x + 0.01*max_x
    min_x = plot_df.i.min() - 0.01*max_x
    plt.xlim([min_x, max_x])
    plot_ax.set_yticks(range(0, 30, 5))

    plot.set_xlabel('Chromosome', fontdict=font_kwargs)
    plot.set_ylabel('-log10(p)', fontdict=font_kwargs)

    if plot_y_thresh:
        plt.hlines(y=[p_sig_t, suggest_t], xmin=min_x, xmax=max_x, colors=['tab:red', 'k'], linestyles='dashed')
    plt.grid(axis='y')
        # plt.tight_layout()

    sns.scatterplot(plot_df, x='i', y='-log10', hue=hue_key, palette=manhatti_palette, legend=legend, linewidth=0.3, style=style_key, ax=outlier_ax)
    y_max_outliers = outlier_ax.get_ylim()[1]
    y_shift = y_max_outliers + y_max_outliers*0.1
    outlier_ax.set_ylim(bottom=plot_break[1], top=y_shift)  # outliers only
    plot_ax.set_ylim(0, plot_break[0])  # most of the data

    outlier_ax.spines["bottom"].set_visible(False)
    plot_ax.spines["top"].set_visible(False)

    # outlier_ax.xaxis.tick_top()
    outlier_ax.tick_params(top=False, bottom=False)

    d = .25  # proportion of vertical to horizontal extent of the slanted line
    ax_kwargs = dict(marker=[(-1, -d), (1, d)], markersize=12,
                linestyle="none", color='k', mec='k', mew=1, clip_on=False)
    outlier_ax.plot([0, 1], [0, 0], transform=outlier_ax.transAxes, **ax_kwargs)
    outlier_ax.set_ylabel('')
    plot_ax.plot([0, 1], [1, 1], transform=plot_ax.transAxes, **ax_kwargs)



    plt.subplots_adjust(hspace=0.05)
    outlier_ax.set_title(title, **font_kwargs)
    plt.tight_layout()

    return plot.figure

In [ ]:
data_base = 'path/to/data/multiallelic/'
savepath = f'{data_base}/out'

if not os.path.exists(savepath):
        os.mkdir(savepath)

In [ ]:
def get_maxes_idxs(df, winsize=250000):
    keep=[]

    for i, r in df.iterrows():
        chr = r['#CHROM']
        pos = r.POS
        cands = df[(df['#CHROM'] == chr) 
                & (abs(df.POS-pos)<winsize)]
        min_idx = cands.P.idxmin()
        if min_idx == i: 
            keep.append(i)

    return keep

# No Rejoins
This is referring to approach 1 in the paper

## Plots/Overview

In [ ]:
biallelics = get_results(f'path/to/initial_str_results_biallelic.csv', cut=False)

In [ ]:
no_rejoined = get_associaTR_results(
    f'{data_base}/associaTR_no_rejoin.tsv',
    rename_chr=True, 
    cut=False, p_name='p_ad_risk_by_proxy', 
).rename(columns=renamer)
no_rejoined.shape

In [ ]:
bi_key = 'biallelic approach'
multi_key = 'multiallelic approach'
biallelics['source'] = bi_key
no_rejoined['source'] = multi_key
no_rejoined_combined = pd.concat([biallelics[biallelics['-log10'] > 2], no_rejoined], ignore_index=True)
manhatti_palette[multi_key] = custom_palette[1]
manhatti_palette[bi_key] = custom_palette[0]
p = broken_manhatti(no_rejoined_combined, 'Comparison of STRs with and without multiallelic splitting', hue_key='source', style_key='source', sort_key='source', legend='auto')

### Custom multi_fit

In [ ]:
def altered_broken_manhatti_multi(df, title, sort_keys=['#CHROM', "POS"], hue_key='#CHROM', style_key=None, legend=None, 
             sort_key='-log10', sort_asc=True, plot_y_thresh=True, do_full=True, suggest_t=5, p_sig_t=-np.log10(gw_t), plot_break=(30,100), kwargs={}, bi_key='biallelic approach'):
    plot_df = df.copy(deep=True)
    f, (outlier_ax, plot_ax) = plt.subplots(2, 1, sharex=True, figsize=(30,10), height_ratios=[1,8])
    if do_full:
        chr_maxxs = plot_df.groupby('#CHROM')['POS'].max()
        chroms_sorted = sorted(chr_maxxs.keys())
        chr_starts = {chroms_sorted[0]: 0}
        for idx, c in enumerate(chroms_sorted[1:], start=1):
            prev_c = chroms_sorted[idx - 1]
            chr_starts[c] = chr_starts[prev_c] + chr_maxxs[prev_c] + 1
        plot_df['i'] = [p + chr_starts[c] for p,c in zip(plot_df['POS'], plot_df['#CHROM'])]
    else:
        plot_df = df.sort_values(sort_keys)
        plot_df = plot_df.reset_index(drop=True); 
        plot_df['i'] = plot_df.index

    plot_df = plot_df.sort_values(by=[sort_key, '-log10'], ascending=[sort_asc, True])
    mask = (plot_df['#CHROM'] % 2 == 0) & (plot_df['source'] == bi_key)
    plot = sns.scatterplot(plot_df[mask], x='i', y='-log10', color='cornflowerblue', legend=None, linewidth=0.3, style=style_key, ax=plot_ax, **kwargs)

    mask = (plot_df['#CHROM'] % 2 == 1) | (plot_df['source'] != bi_key)
    plot = sns.scatterplot(plot_df[mask], x='i', y='-log10', hue=hue_key, palette=manhatti_palette, legend=None, linewidth=0.3, style=style_key, ax=plot_ax, **kwargs)

    labels_df=plot_df.groupby(sort_keys[0])['i'].agg(['median', 'min', 'max'])
    
    labels_df['mid'] = (labels_df['max'].shift() + labels_df['max']) / 2
    labels_df.loc[labels_df['mid'].isna(), 'mid'] = labels_df['max'].iloc[0] / 2

    plot_ax.set_xticks(labels_df['max'])
    plot_ax.set_xticklabels([], minor=False)

    plot_ax.set_xticks(labels_df['mid'], minor=True)
    plot_ax.set_xticklabels(labels_df.index, minor=True)
    plot_ax.xaxis.grid(True)
    
    outlier_ax.set_xticks(labels_df['max'])
    outlier_ax.set_xticklabels([], minor=False)

    outlier_ax.set_xticks(labels_df['mid'], minor=True)
    outlier_ax.set_xticklabels(labels_df.index, minor=True)
    outlier_ax.xaxis.grid(True)

    max_x = plot_df.i.max()
    max_x = max_x + 0.01*max_x
    min_x = plot_df.i.min() - 0.01*max_x
    plt.xlim([min_x, max_x])
    plot_ax.set_yticks(range(0, 30, 5))

    plot.set_xlabel('Chromosome', fontdict=font_kwargs)
    plot.set_ylabel('-log10(p)', fontdict=font_kwargs)

    if plot_y_thresh:
        plt.hlines(y=[p_sig_t, suggest_t], xmin=min_x, xmax=max_x, colors=['tab:red', 'k'], linestyles='dashed')
    plt.grid(axis='y')
        # plt.tight_layout()

    sns.scatterplot(plot_df, x='i', y='-log10', hue=hue_key, palette=manhatti_palette, legend=legend, linewidth=0.3, style=style_key, ax=outlier_ax)
    y_max_outliers = outlier_ax.get_ylim()[1]
    y_shift = y_max_outliers + y_max_outliers*0.1
    outlier_ax.set_ylim(bottom=plot_break[1], top=y_shift)  # outliers only
    plot_ax.set_ylim(0, plot_break[0])  # most of the data

    outlier_ax.spines["bottom"].set_visible(False)
    plot_ax.spines["top"].set_visible(False)

    # outlier_ax.xaxis.tick_top()
    outlier_ax.tick_params(top=False, bottom=False)

    d = .25  # proportion of vertical to horizontal extent of the slanted line
    ax_kwargs = dict(marker=[(-1, -d), (1, d)], markersize=12,
                linestyle="none", color='k', mec='k', mew=1, clip_on=False)
    outlier_ax.plot([0, 1], [0, 0], transform=outlier_ax.transAxes, **ax_kwargs)
    outlier_ax.set_ylabel('')
    plot_ax.plot([0, 1], [1, 1], transform=plot_ax.transAxes, **ax_kwargs)



    plt.subplots_adjust(hspace=0.05)
    outlier_ax.set_title(title, **font_kwargs)
    plt.tight_layout()

    return plot.figure

In [ ]:
bi_key = 'biallelic approach'
multi_key = 'multiallelic approach'
biallelics['source'] = bi_key
no_rejoined['source'] = multi_key
# no_rejoined_combined = pd.concat([biallelics[biallelics['-log10'] > 2], no_rejoined], ignore_index=True)
no_rejoined_combined = pd.concat([biallelics, no_rejoined], ignore_index=True)
manhatti_palette[multi_key] = custom_palette[1]
manhatti_palette[bi_key] = custom_palette[0]
p = altered_broken_manhatti_multi(no_rejoined_combined, 'Comparison of STRs with and without multiallelic splitting', hue_key='source', style_key='source', sort_key='source', legend='auto')

## Find Hits

In [ ]:
simple_merged = pd.merge(biallelics, no_rejoined, on=['#CHROM', 'POS'], suffixes=('_biallelic', '_multi'), how='outer')
simple_merged.keys()

In [ ]:
show_cols = ['#CHROM', 'POS', '-log10_multi', '-log10_biallelic', 'motif', 'ref_len', 'alleles', 'REF', 'ALT', 'islocmax', 'P_multi', 'P_biallelic', 'n_samples_tested', 'OBS_CT']

In [ ]:
mask = (simple_merged.islocmax == 1) & (simple_merged.P_biallelic < gw_t)
save_df = simple_merged[mask][show_cols].sort_values(by=['#CHROM', 'POS', 'P_multi']).reset_index(drop=True)
save_df.to_excel(f"{savepath}/output.xlsx", index=False, sheet_name='gw_tophits-initialdata')
save_df

## Find next to hits

In [ ]:
mask = (biallelics.islocmax == 1) & (biallelics.P < gw_t)
bi = biallelics[mask].copy(deep=True)
muls = no_rejoined.copy(deep=True)
bi['POS_bi'] = bi['POS']
muls['POS_multi'] = muls['POS']
joined_not_hits = pd.merge_asof(
    muls.sort_values(by=['POS']), bi[mask].sort_values(by=['POS']), 
    by='#CHROM', on='POS', suffixes=('_multi', '_bi'),
    allow_exact_matches=False, tolerance=10, direction='nearest'
)
print(mask.value_counts())
joined_not_hits.shape

In [ ]:
joined_not_hits.dropna(inplace=True)

In [ ]:
show_cols = ['#CHROM', 'POS', '-log10_multi', '-log10_bi', 'motif', 'ref_len', 'alleles', 'REF', 'ALT','islocmax', 'P_multi', 'P_bi', 'POS_bi', 'POS_multi', 'posdiff']
joined_not_hits['posdiff'] = abs(joined_not_hits['POS_multi'] - joined_not_hits['POS_bi'])
save_df = joined_not_hits[show_cols].sort_values(by=['#CHROM', 'POS', 'P_multi']).reset_index(drop=True)
# with pd.ExcelWriter(f"{savepath}/output.xlsx", mode='a') as writer:
    # save_df.to_excel(writer, index=False, sheet_name='gw_tophits-nearby_hits')
save_df

In [ ]:
mask = ((save_df['#CHROM'] == 15) & (save_df['POS'] == 63139195)) | ((save_df['#CHROM'] == 19) & (save_df['POS'] == 45802825))
save_df[mask]

# 2-Ways Rejoins
This is referring to approach b in the paper. 

In [ ]:
rejoin_biallels = get_associaTR_results(
    f'{data_base}/2_step_associaTRs.tsv',
    rename_chr=True, 
    cut=False, p_name='p_ad_risk_by_proxy', 
).rename(columns=renamer)
rejoin_biallels.shape

In [ ]:
biallelics = get_results(f'{data_base}/../initial_str_results_biallelic.csv', cut=False)

In [ ]:
biallelics['source'] = 'multiallelic splitting'
rejoin_biallels['source'] = 'rejoining to multiallelic'
manhatti_palette['multiallelic splitting'] = custom_palette[0]
manhatti_palette['rejoining to multiallelic'] = custom_palette[1]

rejoined_combined = pd.concat([biallelics[biallelics['-log10'] > 2.5], rejoin_biallels], ignore_index=True)
p = broken_manhatti(rejoined_combined, 'Comparison of STRs that were rejoined and multiallelic splitting', hue_key='source', style_key='source', sort_key='source', legend='auto')

### Find hits

In [ ]:
simple_merged = pd.merge(biallelics, rejoin_biallels, on=['#CHROM', 'POS'], suffixes=('_biallelic', '_multi'), how='outer')
simple_merged.keys()

In [ ]:
show_cols = ['#CHROM', 'POS', '-log10_multi', '-log10_biallelic', 'motif', 'ref_len', 'alleles', 'REF', 'ALT', 'islocmax', 'P_multi', 'P_biallelic', 'n_samples_tested', 'OBS_CT']

In [ ]:
mask = (simple_merged.islocmax == 1) & (simple_merged.P_biallelic < gw_t)
save_df = simple_merged[mask][show_cols].sort_values(by=['#CHROM', 'POS']).reset_index(drop=True)
with pd.ExcelWriter(f"{savepath}/output.xlsx", mode='a', if_sheet_exists='replace') as writer:
    save_df.to_excel(writer, index=False, sheet_name='gw_tophits-joined_data')
save_df

## Find next to hits

In [ ]:
mask = (biallelics.islocmax == 1) & (biallelics.P < gw_t)
bi = biallelics[mask].copy(deep=True)
muls = rejoin_biallels.copy(deep=True)
bi['POS_bi'] = bi['POS']
muls['POS_multi'] = muls['POS']
joined_not_hits = pd.merge_asof(
    muls.sort_values(by=['POS']), bi[mask].sort_values(by=['POS']), 
    by='#CHROM', on='POS', suffixes=('_multi', '_bi'),
    allow_exact_matches=False, tolerance=10, direction='nearest'
)
print(mask.value_counts())
joined_not_hits.shape

In [ ]:
joined_not_hits.dropna(inplace=True)

In [ ]:
show_cols = ['#CHROM', 'POS', '-log10_multi', '-log10_bi', 'motif', 'ref_len', 'alleles', 'REF', 'ALT','islocmax', 'P_multi', 'P_bi', 'POS_bi', 'POS_multi', 'posdiff']
joined_not_hits['posdiff'] = abs(joined_not_hits['POS_multi'] - joined_not_hits['POS_bi'])
save_df = joined_not_hits[show_cols].sort_values(by=['#CHROM', 'POS', 'P_multi']).reset_index(drop=True)
with pd.ExcelWriter(f"{savepath}/output.xlsx", mode='a', if_sheet_exists='replace') as writer:
    save_df.to_excel(writer, index=False, sheet_name='gw_tophits-nearby_hits')
save_df

## hits in multi_regions

In [ ]:
max_idxs = get_maxes_idxs(rejoin_biallels, winsize=250000)
rejoin_biallels['islocmax'] = np.where(rejoin_biallels.index.isin(max_idxs), 1, 0)
rejoin_biallels.islocmax.value_counts()

In [ ]:
merged2 = pd.merge(rejoin_biallels, biallelics, on=['#CHROM', 'POS'], suffixes=('_multi', '_bi'), how='outer')
merged2.shape

In [ ]:
merged2.head()

In [ ]:
mask = ((~merged2.motif.isna().astype(bool)) & (merged2.islocmax_multi == 1) & (merged2.P_multi < 1e-5)) | ((merged2.islocmax_bi == 1) & (merged2.P_bi < 1e-5))
both_df = merged2[mask].copy(deep=True)
both_df.shape

In [ ]:
show_cols = ['#CHROM', 'POS', '-log10_multi', '-log10_bi', 'motif', 'ref_len', 'alleles', 'REF', 'islocmax_multi', 'islocmax_bi', 'P_multi', 'P_bi']

In [ ]:
mask = (both_df.islocmax_multi != both_df.islocmax_bi) & ((both_df.P_multi < gw_t) | (both_df.P_bi < gw_t))
both_df[mask][show_cols].reset_index(drop=True)

In [ ]:
mask = (both_df.P_multi < gw_t) & (both_df.islocmax_multi != both_df.islocmax_bi)
both_df[mask][show_cols].reset_index(drop=True)

In [ ]:
mask = (both_df.P_multi < gw_t) | ((both_df.P_bi < suggestive_t) & (both_df.islocmax_bi == 1)) 
save_df = both_df[mask][show_cols].sort_values(by=['#CHROM', 'POS']).reset_index(drop=True)
with pd.ExcelWriter(f"{savepath}/output.xlsx", mode='a', if_sheet_exists='replace') as writer:
    save_df.to_excel(writer, index=False, sheet_name='all_hits')
save_df

# Note: Variants that were compared
We ran the multiallelic tests only in regions around our bi-alleic top hits. The following plot illustrates the variants in bi-allelic vs multi-allelic that were actually compared. 

In [ ]:
initials = get_results(f'{data_base}/../initial_str_results_biallelic.csv', cut=False)
bounds = pd.read_csv('path/to/data/multiallelic/extract_regions.bed', sep='\t', names=['chr', 'lower', 'upper']) 
print(bounds.shape)
bounds.head()

In [ ]:
def check_bounds(row, bounds):
    chr = row['#CHROM']
    pos = row['POS']
    b = bounds[bounds['chr'] == f'chr{chr}']
    if b.shape[0] == 0:
        return False
    for _, r in b.iterrows():
        if (pos >= r['lower']) & (pos <= r['upper']):
            return True
    return False

In [ ]:
founds = pd.DataFrame()
for i,r in bounds.iterrows():
    print(i, r)
    chr = bounds.loc[i, 'chr'].replace('chr', '')
    l = bounds.loc[i, 'lower']
    u = bounds.loc[i, 'upper']

    mask = (initials['#CHROM'] == int(chr)) & (initials['POS'] >= l) & (initials['POS'] <= u)
    founds = pd.concat([founds, initials[mask]])

In [ ]:
rechecked = get_associaTR_results(
    f'{data_base}/2_step_associaTRs.tsv',
    rename_chr=True, 
    cut=False, p_name='p_ad_risk_by_proxy', 
).rename(columns=renamer)

In [ ]:
founds['source'] = 'initial biallelic'
rechecked['source'] = 'rechecked with multiallelic'
show_combined = pd.concat([founds, rechecked], ignore_index=True)

In [ ]:
manhatti_palette['initial biallelic'] = custom_palette[0]
manhatti_palette['rechecked with multiallelic'] = custom_palette[1]
p = broken_manhatti(show_combined, 'Actually compared variants', hue_key='source', style_key='source', sort_key='-log10', legend='auto')
